In [13]:
# # Parameters cell
# Month = "2025-01-01"
# top_n = 10
# Scenario = "Actuals"

# Set variables if not defined
try:
    Month
except NameError:
    Month = "2025-01-01"

try:
    top_n
except NameError:
    top_n = 10

try:
    Scenario
except NameError:
    Scenario = "Actuals"

In [14]:
%%tsql -artifact CowiWarehouse -type Warehouse -session

SELECT TOP({top_n}) * FROM [CowiWarehouse].[dbo].[Forecast] WHERE Scenario = '{Scenario}' AND [Month] >= '{Month}';

In [35]:
%%tsql -artifact CowiWarehouse -type Warehouse -session

INSERT INTO [dbo].[Forecast]
    ([Project Key], [Month], [Scenario], [Item], [FTE], [Allocated Hours], [Hour Rate avg DKK], [Sales DKK], [Status])
VALUES
    (6, '{Month}', '{Scenario}', NULL, '2.5', '136.0', '150.0', '51000.0',  'Fulfilled'),
    (6, '{Month}', '{Scenario}', NULL, '3.0', '140.0', '151.0', '63420.0',  'Fulfilled')
;

In [18]:
df = %tsql SELECT TOP({top_n}) * FROM [CowiWarehouse].[dbo].[Forecast] WHERE Scenario = '{Scenario}' AND [Month] >= '{Month}';

In [16]:
print (df)

   Project Key       Month Scenario  Item     FTE Allocated Hours  \
0            6  2025-01-01  Actuals  None  2.5000        136.0000   
1            6  2025-02-01  Actuals  None  3.0000        140.0000   
2            6  2025-03-01  Actuals  None  3.5000        144.0000   
3            6  2025-06-01  Actuals  None  5.0000        136.0000   
4            6  2025-04-01  Actuals  None  4.0000        128.0000   
5            6  2025-05-01  Actuals  None  4.5000        132.0000   
6            6  2025-09-01  Actuals  None  3.0000        128.0000   
7            6  2025-08-01  Actuals  None  2.5000        144.0000   
8            6  2025-07-01  Actuals  None  2.0000        100.0000   
9            2  2025-01-01  Actuals  None  8.5000        140.0000   

  Hour Rate avg DKK    Sales DKK     Status  
0          150.0000   51000.0000  Fulfilled  
1          151.0000   63420.0000  Fulfilled  
2          149.0000   75096.0000  Fulfilled  
3          149.0000  101320.0000  Fulfilled  
4         

In [39]:
import pyarrow as pa

conn = notebookutils.data.connect_to_artifact("CowiWarehouse", "f05ea07b-6c22-48cb-9fa0-e4e73cb78b7d", "Warehouse")
sql = """
SELECT TOP (100) [Project Key],
			[Month],
			[Scenario],
			[Item],
			[FTE],
			[Allocated Hours],
			[Hour Rate avg DKK],
			[Sales DKK],
			[Status]
FROM [CowiWarehouse].[dbo].[Forecast] WHERE Scenario = 'Jeffrey3'
"""
cursor = conn.execute(sql)
columns = [column[0] for column in cursor.description]
data = cursor.fetchall()
columnar_data = list(zip(*data))
if len(columnar_data) > 0:
	arrow_table = pa.Table.from_arrays([pa.array(col) for col in columnar_data], columns)
	display(arrow_table.to_pandas())
else:
	print('empty table')
